# Section 1: Email Security and Phishing Detection

This notebook demonstrates the complete assignment workflow by comparing:

1. **Classic machine learning:** TF-IDF with Logistic Regression
2. **Deep learning:** a token-embedding LSTM

The Apache SpamAssassin public corpus is used because it is explicitly suggested in the assignment brief.

> **Validity warning:** SpamAssassin is labelled **spam versus ham**, not **phishing versus legitimate email**. This notebook demonstrates the required text-classification method, but its scores must not be presented as phishing-specific performance.


## Before running

When the notebook uses the **Google Colab** kernel in VS Code, the notebook file remains in this local repository but code executes on a temporary remote machine.

For a remote run:

1. Connect the notebook to **Kernel -> Colab -> Auto Connect** in VS Code.
2. Run the cells from top to bottom. The setup cell automatically clones `https://github.com/Saumyakeshi/ml_assignment.git` into `/content/ml_assignment` when required.
3. Do not upload `.venv`, raw data, or trained models manually. The notebook downloads its dataset and dependencies.
4. For the final experiment, set `SMOKE_TEST = False` in the LSTM cell.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Saumyakeshi/ml_assignment.git"
COLAB_PROJECT_DIR = Path("/content/ml_assignment")


def is_project(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (path / "src" / "comp70049").is_dir()


def find_local_project() -> Path | None:
    current = Path.cwd().resolve()
    return next((candidate for candidate in [current, *current.parents] if is_project(candidate)), None)


running_on_colab = Path("/content").is_dir()
if running_on_colab:
    if not is_project(COLAB_PROJECT_DIR):
        if COLAB_PROJECT_DIR.exists():
            raise FileExistsError(
                f"{COLAB_PROJECT_DIR} exists but is not a complete project. "
                "Remove or rename it, then rerun this cell."
            )
        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(COLAB_PROJECT_DIR)],
            check=True,
        )
    project_dir = COLAB_PROJECT_DIR
else:
    project_dir = find_local_project()
    if project_dir is None:
        raise FileNotFoundError(
            "Could not find pyproject.toml and src/comp70049. "
            "Open the notebook from inside the repository."
        )

os.chdir(project_dir)
source_dir = project_dir / "src"
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
print("Execution environment:", "Google Colab" if running_on_colab else "Local")
print("Project directory:", Path.cwd())
print("Python source directory:", source_dir)


## Install the project

This installs the repository as an editable Python package in the active notebook kernel. Colab runtimes are temporary, so this cell should remain in the notebook.


In [ ]:
%pip install -q -e ".[deep]"


In [ ]:
import platform

import sklearn
import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Dataset provenance

Source: [Apache SpamAssassin public corpus](https://spamassassin.apache.org/old/publiccorpus/)

Selected archives:

- `20030228_easy_ham.tar.bz2`
- `20030228_spam.tar.bz2`

The loader combines each email's subject and readable MIME body, ignores attachments, removes exact text duplicates, and assigns `0 = ham`, `1 = spam`.


In [ ]:
subprocess.run(
    [sys.executable, "scripts/download_spamassassin.py"],
    check=True,
)


In [ ]:
import json
import sys
from pathlib import Path

source_dir = Path.cwd() / "src"
if not (source_dir / "comp70049").is_dir():
    raise FileNotFoundError(
        f"Missing {source_dir / 'comp70049'}. Upload the complete src "
        "folder to the Colab runtime and rerun the setup cell."
    )
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from comp70049.phishing.data import (
    load_spamassassin_corpus,
    save_split_manifest,
    stratified_splits,
)

# Central configuration keeps the notebook and command-line experiment aligned.
config = json.loads(Path("configs/section_01.json").read_text(encoding="utf-8"))
emails = load_spamassassin_corpus(Path(config["data_dir"]))

label_names = {0: "ham", 1: "spam"}
class_counts = emails["label"].map(label_names).value_counts()

print("Messages after exact deduplication:", len(emails))
display(class_counts.rename("messages").to_frame())

axis = class_counts.reindex(["ham", "spam"]).plot.bar(
    color=["#4C78A8", "#E45756"],
    title="SpamAssassin class distribution",
    ylabel="Messages",
    rot=0,
)
axis.grid(axis="y", alpha=0.25)
plt.show()


In [ ]:
analysis_frame = emails.assign(
    class_name=emails["label"].map(label_names),
    character_count=emails["text"].str.len(),
    word_count=emails["text"].str.split().str.len(),
)

display(
    analysis_frame.groupby("class_name")[["character_count", "word_count"]]
    .agg(["mean", "median", "min", "max"])
    .round(1)
)

display(
    analysis_frame[["class_name", "text"]]
    .sample(3, random_state=config["seed"])
    .assign(text=lambda frame: frame["text"].str.slice(0, 500))
)


## 2. Leakage-safe data splitting

The split is performed after exact deduplication:

- 70% training
- 15% validation
- 15% test

Both models use the same held-out test records. The TF-IDF statistics and LSTM vocabulary are learned only from the training split.


In [ ]:
# Deduplication occurs inside the loader before this stratified split, which
# prevents identical emails from appearing in more than one partition.
splits = stratified_splits(
    emails,
    test_fraction=config["test_fraction"],
    validation_fraction=config["validation_fraction"],
    seed=config["seed"],
)

split_manifest = save_split_manifest(
    splits,
    Path(config["processed_dir"]),
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "records": [
            len(splits.train),
            len(splits.validation),
            len(splits.test),
        ],
        "spam_rate": [
            splits.train["label"].mean(),
            splits.validation["label"].mean(),
            splits.test["label"].mean(),
        ],
    }
)

display(split_summary.style.format({"spam_rate": "{:.2%}"}))
print("Split manifest:", split_manifest)


## 3. Text preprocessing

The shared normalizer:

- converts text to lowercase;
- replaces URLs with `urltoken`;
- replaces email addresses with `emailtoken`;
- removes punctuation;
- collapses repeated whitespace.

The classic pipeline additionally removes English stopwords and creates TF-IDF unigram and bigram features.


In [ ]:
from comp70049.phishing.preprocessing import normalize_text

example_text = (
    "URGENT: Contact Support@Example.com and verify your account at "
    "https://example.com/login!"
)

print("Original:  ", example_text)
print("Normalized:", normalize_text(example_text))


## 4. Classic model: TF-IDF and Logistic Regression

The vectorizer and classifier are joined in a scikit-learn pipeline. This ensures that TF-IDF is fitted only on the training set and then applied unchanged to the test set.


In [ ]:
from comp70049.phishing.classic import (
    evaluate_classic_model,
    train_classic_model,
)

models_dir = Path(config["models_dir"])
results_dir = Path(config["results_dir"])

# The pipeline learns TF-IDF statistics only from the training records.
classic_model = train_classic_model(
    splits.train,
    config["classic"],
)

classic_metrics = evaluate_classic_model(
    classic_model,
    splits.test,
    model_dir=models_dir,
    results_dir=results_dir,
)

display(
    pd.Series(
        {
            key: classic_metrics[key]
            for key in (
                "accuracy",
                "precision",
                "recall",
                "f1",
                "roc_auc",
                "average_precision",
            )
        },
        name="TF-IDF Logistic Regression",
    ).to_frame("score").style.format("{:.4f}")
)


In [ ]:
for figure_name in (
    "tf-idf-logistic-regression-confusion-matrix.png",
    "tf-idf-logistic-regression-roc-curve.png",
    "tf-idf-logistic-regression-precision-recall-curve.png",
):
    display(Image(filename=str(results_dir / "figures" / figure_name)))


## 5. Deep-learning model: LSTM

The LSTM uses:

- a vocabulary learned only from training messages;
- a learned embedding layer;
- an LSTM sequence encoder;
- dropout and a binary output layer;
- weighted binary cross-entropy to address class imbalance;
- validation F1 for early stopping.

A one-epoch smoke test proves that the pipeline runs. It is **not** sufficient for the final model comparison.


In [ ]:
from comp70049.phishing.lstm import train_and_evaluate_lstm

# Keep this True for a quick end-to-end check.
# Set it to False before producing the final assignment results.
SMOKE_TEST = True

lstm_config = config["lstm"].copy()

if SMOKE_TEST:
    lstm_config["epochs"] = 1

print("LSTM epochs requested:", lstm_config["epochs"])

lstm_metrics = train_and_evaluate_lstm(
    splits.train,
    splits.validation,
    splits.test,
    config=lstm_config,
    seed=config["seed"],
    model_dir=models_dir,
    results_dir=results_dir,
)

display(
    pd.Series(
        {
            key: lstm_metrics[key]
            for key in (
                "accuracy",
                "precision",
                "recall",
                "f1",
                "roc_auc",
                "average_precision",
                "best_validation_f1",
                "epochs_completed",
            )
        },
        name="LSTM",
    ).to_frame("value")
)


In [ ]:
for figure_name in (
    "lstm-training-history.png",
    "lstm-confusion-matrix.png",
    "lstm-roc-curve.png",
    "lstm-precision-recall-curve.png",
):
    display(Image(filename=str(results_dir / "figures" / figure_name)))


## 6. Direct model comparison

Both models are compared on the same test set using accuracy, precision, recall, F1, ROC AUC, and average precision. Because the classes are imbalanced, accuracy should not be interpreted alone.


In [ ]:
metric_names = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
]

# Use the same test records and metrics for a fair side-by-side comparison.
comparison = pd.DataFrame(
    [
        {
            "model": "TF-IDF Logistic Regression",
            **{name: classic_metrics[name] for name in metric_names},
        },
        {
            "model": "LSTM",
            **{name: lstm_metrics[name] for name in metric_names},
        },
    ]
)

comparison.to_csv(results_dir / "model-comparison.csv", index=False)
display(
    comparison.style
    .format({name: "{:.4f}" for name in metric_names})
    .highlight_max(subset=metric_names, color="#d9ead3")
)


## 7. Interpretation checklist

Use the observed metrics and confusion matrices to answer:

1. Which model produces fewer false positives?
2. Which model produces fewer false negatives?
3. Does the LSTM improve enough to justify its training cost and lower explainability?
4. How does class imbalance affect the usefulness of accuracy?
5. What would happen if the models were applied to modern phishing emails?
6. How should a production threshold reflect the different costs of false positives and false negatives?

### Cybersecurity implications

- False positives can block legitimate business communication.
- False negatives allow unwanted or malicious messages to reach users.
- Attackers adapt their wording, creating distribution shift.
- Email content may contain sensitive information and requires privacy controls.
- A production detector should be monitored, recalibrated, and retrained.

### Limitations

- SpamAssassin is spam-labelled rather than phishing-labelled.
- The corpus is old and may contain collection-specific artifacts.
- Easy ham may make the task artificially simple.
- A one-epoch LSTM is only a pipeline smoke test.
- Exact deduplication does not remove near-duplicate campaign messages.


## 8. Optional Google Drive export

The following cell does nothing until `SAVE_TO_DRIVE` is changed to `True`. Mounting Drive gives the notebook access to your Drive files, so run it only when you want to preserve the remote artifacts.


In [ ]:
# Opt in explicitly because mounting Drive grants access to personal files.
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from datetime import datetime
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    export_root = (
        Path("/content/drive/MyDrive/COMP70049")
        / f"section_01_results_{timestamp}"
    )
    export_root.mkdir(parents=True, exist_ok=False)

    shutil.copytree(results_dir, export_root / "reports")
    shutil.copytree(models_dir, export_root / "models")

    print("Results exported to:", export_root)
else:
    print("Drive export disabled. Set SAVE_TO_DRIVE = True to enable it.")


In [ ]:
print("Generated result files:")
for path in sorted(results_dir.rglob("*")):
    if path.is_file():
        print(" -", path)

print("\nGenerated model files:")
for path in sorted(models_dir.rglob("*")):
    if path.is_file():
        print(" -", path)
